# MMDEW TCPD Oracle results

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

START_DIR = Path.cwd().resolve()
NOTEBOOK_NAME = 'MMDEW_kerneldetector_TCPD_VISIBLE_TABLES.ipynb'
HERE = next(
    candidate for parent in (START_DIR, *START_DIR.parents)
    for candidate in (parent, parent / 'code')
    if (candidate / NOTEBOOK_NAME).exists()
)

code_candidates = [HERE, *HERE.parents]
code_candidates.extend(parent / 'OKAFF' for parent in HERE.parents)
CODE_ROOT = next(
    (
        candidate
        for candidate in code_candidates
        if (candidate / 'src' / 'kerneldetector.py').is_file()
    ),
    None,
)
if CODE_ROOT is None:
    raise FileNotFoundError('Could not locate the project src directory')

SRC_DIR = CODE_ROOT / 'src'
for module_dir in (HERE, SRC_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import kerneldetector

if Path(kerneldetector.__file__).resolve().parent != SRC_DIR.resolve():
    raise RuntimeError('Restart the kernel so kerneldetector loads from src')

from kerneldetector import MMDEW, GaussianKernel
from metrics import f_measure

DATASET_DIR = HERE / 'datasets'
ANNOTATIONS_FILE = HERE / 'annotations.json'
RESULTS = HERE / 'results' / 'MMDEW_kerneldetector_TCPD'
RESULTS.mkdir(parents=True, exist_ok=True)

REFERENCE_SIZE = 50
REFERENCE_THRESHOLD_ALPHA = 0.10
FIRST_ALARM_POSITION = 50
MATCH_MARGIN = 5
EVALUATION_START = FIRST_ALARM_POSITION - MATCH_MARGIN
ORACLE_RHO_GRID = (0.005, 0.01, 0.05, 0.10, 0.20)
ORACLE_Q_GRID = (0.90, 0.95, 0.99, 0.995)
MMDEW_SEED = 1234


## Oracle grid search and tables

In [2]:
EXCLUDED = {f'quality_control_{i}' for i in range(1, 6)} | {'uk_coal_employ'}

with ANNOTATIONS_FILE.open() as stream:
    annotations = json.load(stream)


def load_dataset(name):
    with (DATASET_DIR / f'{name}.json').open() as stream:
        metadata = json.load(stream)
    matrix = np.asarray([
        [np.nan if value is None else value for value in series['raw']]
        for series in metadata['series']
    ], dtype=float).T
    if not np.all(np.isfinite(matrix)):
        raise ValueError(f'{name} contains non-finite observations')
    return metadata, matrix


datasets = sorted(
    name for name in annotations
    if name not in EXCLUDED
    and json.loads((DATASET_DIR / f'{name}.json').read_text())['n_obs'] > REFERENCE_SIZE
)
metadata_by_name, matrices = {}, {}
for name in datasets:
    metadata_by_name[name], matrices[name] = load_dataset(name)


def retained_annotation(name):
    return {
        annotator: [point for point in points if point >= EVALUATION_START]
        for annotator, points in annotations[name].items()
    }


def calculate_mmdew_statistic(name):
    matrix = matrices[name]
    gamma = float(GaussianKernel.est_gamma(matrix[:REFERENCE_SIZE]))
    detector = MMDEW(
        gamma=gamma,
        alpha=0.01,
        min_elements_per_window=1,
        max_windows=0,
        cooldown=500,
        seed=MMDEW_SEED,
    )
    for sample in matrix:
        detector.insert(sample)
    return np.asarray(detector.stats, dtype=float)


def adaptive_threshold(statistic, rho, q):
    z_q = norm.ppf(q)
    mu2, mu4 = 0.0, 0.0
    threshold = np.empty(len(statistic), dtype=float)
    for index, value in enumerate(statistic):
        rate = REFERENCE_THRESHOLD_ALPHA if index < REFERENCE_SIZE else rho
        mu2 = (1 - rate) * mu2 + rate * value ** 2
        mu4 = (1 - rate) * mu4 + rate * value ** 4
        variance_term = max(0.0, mu4 - mu2 ** 2)
        threshold[index] = np.sqrt(max(0.0, mu2 + z_q * np.sqrt(variance_term)))
    return threshold


statistics = {name: calculate_mmdew_statistic(name) for name in datasets}


def score_threshold(name, rho, q):
    statistic = statistics[name]
    threshold = adaptive_threshold(statistic, rho, q)
    alarms = np.flatnonzero(
        (statistic > threshold)
        & (np.arange(len(statistic)) >= FIRST_ALARM_POSITION)
    ).tolist()
    return {
        'dataset': name,
        'data type': (
            'multivariate' if metadata_by_name[name]['n_dim'] > 1 else 'univariate'
        ),
        'rho': rho,
        'q': q,
        'F1': f_measure(retained_annotation(name), alarms),
    }


oracle_rows = []
for dataset_number, name in enumerate(datasets, 1):
    print(f'[Oracle {dataset_number:02d}/{len(datasets):02d}] {name}', flush=True)
    for rho in ORACLE_RHO_GRID:
        for q in ORACLE_Q_GRID:
            oracle_rows.append(score_threshold(name, rho, q))

oracle_grid = pd.DataFrame(oracle_rows)
oracle_f1 = oracle_grid.loc[
    oracle_grid.groupby('dataset')['F1'].idxmax()
].sort_values('dataset').reset_index(drop=True)

oracle_summary = (
    oracle_f1.groupby('data type', as_index=False)
    .agg(**{
        'Oracle F1': ('F1', 'mean'),
        'N datasets': ('dataset', 'nunique'),
    })
    [['data type', 'Oracle F1', 'N datasets']]
)
oracle_f1_table = oracle_f1[[
    'dataset', 'data type', 'rho', 'q', 'F1'
]].rename(columns={'F1': 'Oracle F1'})

oracle_summary.to_csv(RESULTS / 'mmdew_oracle_summary.csv', index=False)
oracle_f1_table.to_csv(
    RESULTS / 'mmdew_oracle_f1_by_dataset.csv', index=False
)

print('Overall Oracle results')
display(oracle_summary.round({'Oracle F1': 4}))

aggregate_excluded_datasets = {'gdp_iran', 'gdp_japan', 'ozone', 'robocalls'}
oracle_summary_excluding = (
    oracle_f1.loc[~oracle_f1['dataset'].isin(aggregate_excluded_datasets)]
    .groupby('data type', as_index=False)
    .agg(**{
        'Oracle F1': ('F1', 'mean'),
        'N datasets': ('dataset', 'nunique'),
    })
    [['data type', 'Oracle F1', 'N datasets']]
)
print(
    'Oracle results excluding gdp_iran, gdp_japan, ozone, and robocalls'
)
display(oracle_summary_excluding.round({'Oracle F1': 4}))

print('Oracle-F1: maximum F1 per dataset')
with pd.option_context('display.max_rows', None):
    display(oracle_f1_table.round(4))


[Oracle 01/32] apple
[Oracle 02/32] bank
[Oracle 03/32] bee_waggle_6
[Oracle 04/32] bitcoin
[Oracle 05/32] brent_spot
[Oracle 06/32] businv
[Oracle 07/32] children_per_woman
[Oracle 08/32] co2_canada
[Oracle 09/32] construction
[Oracle 10/32] gdp_argentina
[Oracle 11/32] gdp_iran
[Oracle 12/32] gdp_japan
[Oracle 13/32] global_co2
[Oracle 14/32] homeruns
[Oracle 15/32] iceland_tourism
[Oracle 16/32] jfk_passengers
[Oracle 17/32] lga_passengers
[Oracle 18/32] measles
[Oracle 19/32] nile
[Oracle 20/32] occupancy
[Oracle 21/32] ozone
[Oracle 22/32] ratner_stock
[Oracle 23/32] robocalls
[Oracle 24/32] run_log
[Oracle 25/32] scanline_126007
[Oracle 26/32] scanline_42049
[Oracle 27/32] seatbelts
[Oracle 28/32] shanghai_license
[Oracle 29/32] unemployment_nl
[Oracle 30/32] us_population
[Oracle 31/32] usd_isk
[Oracle 32/32] well_log
Overall Oracle results


,data type,Oracle F1,N datasets
0,multivariate,0.7848,4
1,univariate,0.7846,28


Oracle results excluding gdp_iran, gdp_japan, ozone, and robocalls


,data type,Oracle F1,N datasets
0,multivariate,0.7848,4
1,univariate,0.7487,24


Oracle-F1: maximum F1 per dataset


,dataset,data type,rho,q,Oracle F1
0,apple,multivariate,0.100,0.995,0.6957
1,bank,univariate,0.005,0.990,1.0000
2,bee_waggle_6,multivariate,0.005,0.990,0.9286
3,bitcoin,univariate,0.200,0.990,0.4496
4,brent_spot,univariate,0.200,0.950,0.3392
5,businv,univariate,0.005,0.900,0.5882
6,children_per_woman,univariate,0.100,0.990,0.5075
7,co2_canada,univariate,0.200,0.950,0.6905
8,construction,univariate,0.100,0.995,0.6957
9,gdp_argentina,univariate,0.050,0.900,1.0000
